In [1]:
# !pip install --upgrade --force-reinstall opencv-python
# !pip install keras_preprocessing

Defaulting to user installation because normal site-packages is not writeable
  Using cached opencv_python-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached numpy-1.24.4-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
Using cached opencv_python-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (63.0 MB)
Using cached numpy-1.24.4-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
^C
ERROR: Operation cancelled by user

[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip


In [1]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report
import os
import pandas as pd

# from data_pre_proc import load_data
# from Train import get_model, r2_keras

tf.compat.v1.enable_eager_execution()
from tensorflow.compat.v1 import ConfigProto, InteractiveSession
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Dropout, Flatten, Dense
from tensorflow.keras.activations import relu
from tensorflow.keras.initializers import he_normal, zeros, RandomNormal
from tensorflow.keras.models import Model
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras import backend as keras_be
from keras_preprocessing.image import img_to_array
from tqdm import tqdm

2025-01-31 16:37:44.837339: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-31 16:37:44.940001: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-31 16:37:45.570748: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/jovyan/.local/lib/python3.8/site-packages/cv2/../../lib64:/usr/local/lib:/usr/loc

In [2]:
def load_image_mask_subset(path):
    image_dir = os.path.join(path, "images")
    if not os.path.exists(image_dir):
        raise FileNotFoundError(f"Directory not found: {image_dir}")
    file_name_image = os.listdir(image_dir)
    print(f'Loading images from {os.path.join(path, "images")}')
    num_samples = len(file_name_image)

    image_list = []
    file_name_list = []

    image_counter_limit = 6000
    image_counter = 0

    for file_name in file_name_image:
        print(f'Loading {file_name}')
        file_path = os.path.join(path, "images", file_name)

        if not os.path.isfile(file_path):
            continue

        image = cv2.imread(file_path)
        if image is None:
            print(f'Warning: Unable to read {file_name}')
            continue

        # yield image, file_name

        image_list.append(image)
        file_name_list.append(file_name)
        # del image
        image_counter += 1
        if image_counter >= image_counter_limit:
            break

    print('loaded images!')

    return image_list, num_samples, file_name_image

In [3]:
def get_model(img_size=(30, 30)):
    """
    Construct and return model
    """
    input_layer = Input(shape=(img_size[0], img_size[1], 3,), dtype='float32')
    cv1 = Conv2D(filters=32, kernel_size=5, activation=relu,
                 kernel_initializer=he_normal(), bias_initializer=zeros())(input_layer)
    cv2 = Conv2D(filters=64, kernel_size=3, activation=relu,
                 kernel_initializer=he_normal(), bias_initializer=zeros())(cv1)
    mp1 = MaxPool2D(pool_size=(2, 2))(cv2)
    do1 = Dropout(0.25)(mp1)
    cv3 = Conv2D(filters=64, kernel_size=3, activation=relu,
                 kernel_initializer=he_normal(), bias_initializer=zeros())(do1)
    mp2 = MaxPool2D(pool_size=(2, 2))(cv3)
    do2 = Dropout(0.25)(mp2)

    flat = Flatten()(do2)
    fc1 = Dense(units=256, activation=relu, kernel_initializer=he_normal(),
                bias_initializer=zeros(), kernel_regularizer=l2(1e-3))(flat)
    do3 = Dropout(0.5)(fc1)

    # Outputs
    cf = Dense(units=43, activation=None, name="classification",
               kernel_regularizer=l2(1e-4))(do3)
    reg = Dense(units=4, activation='linear', name="regression",
                kernel_initializer=RandomNormal(), kernel_regularizer=l2(0.1))(do3)

    return Model(inputs=input_layer, outputs=[cf, reg])


def r2_keras(y_true, y_pred):
    """
    Coefficient of determination for regression model
    """
    ss_res = keras_be.sum(keras_be.square(y_true - y_pred))
    ss_tot = keras_be.sum(keras_be.square(y_true - keras_be.mean(y_true)))
    return 1 - ss_res / (ss_tot + keras_be.epsilon())

In [4]:
def preprocess_image(image_path, img_size=(30, 30)):
    """Preprocess an image similar to read_data function."""
    image = cv2.imread(image_path)
    if image is None:
        return None
    image = cv2.resize(image, img_size)
    image = img_to_array(image) / 255  # Normalize to [0,1]
    return image

In [5]:
weights_folder = "data"
test_data_folder = "data/test"
test_images, num_samples_test, file_name_test_images = load_image_mask_subset(test_data_folder)

# initialize model
model = get_model((100, 100))
loss = SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer="adam", loss={"classification": loss, "regression": "mse"},metrics={"classification": "acc", "regression": r2_keras},
              loss_weights={"classification": 5, "regression": 1})



Loading images from data/test/images
Loading patched_0.png
Loading patched_1.png
Loading patched_2.png
Loading patched_3.png
Loading patched_4.png
Loading patched_5.png
Loading patched_6.png
Loading patched_7.png
Loading patched_8.png
Loading patched_9.png
Loading patched_10.png
Loading patched_11.png
Loading patched_12.png
Loading patched_13.png
Loading patched_14.png
Loading patched_15.png
Loading patched_16.png
Loading patched_17.png
Loading patched_18.png
Loading patched_19.png
Loading patched_20.png
Loading patched_21.png
Loading patched_22.png
Loading patched_23.png
Loading patched_24.png
Loading patched_25.png
Loading patched_26.png
Loading patched_27.png
Loading patched_28.png
Loading patched_29.png
Loading patched_30.png
Loading patched_31.png
Loading patched_32.png
Loading patched_33.png
Loading patched_34.png
Loading patched_35.png
Loading patched_36.png
Loading patched_37.png
Loading patched_38.png
Loading patched_39.png
Loading patched_40.png
Loading patched_41.png
Loading

2025-01-31 16:38:20.895958: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13598 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:3b:00.0, compute capability: 7.5
2025-01-31 16:38:20.898432: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13598 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:d8:00.0, compute capability: 7.5


In [6]:
# load pre trained weights
model.load_weights("data/result.weights.h5")

In [9]:
train_df = pd.read_csv("train.csv")
print(train_df.head())
results = []
test_images_folder = os.path.join(test_data_folder, "images")

for idx in tqdm(range(len(test_images)), desc="Processing Images"):
    image, filename = test_images[idx], file_name_test_images[idx]
    if image is None:
        continue
    image = np.expand_dims(image, axis=0)
    classification_output, _ = model(image, training=False)
    defense_pred = tf.argmax(classification_output, axis=1).numpy()[0]
    matched_row = train_df[train_df["Patched_path"].str.endswith(filename)]
    # print(f'{filename}: {matched_row}')
    if matched_row.empty:
        continue
    original_pred = matched_row["Original_pred"].values[0]
    patched_pred = matched_row["Patched_pred"].values[0]
    results.append({"Filename": filename, "Original_pred": original_pred, "Patched_pred": patched_pred, "Defense_pred": defense_pred})

results_df = pd.DataFrame(results)
print(results_df.head())
results_df.to_csv("defense_predictions.csv", index=False)

   ClassID  Original_pred  Patched_pred                    Mask_path  \
0        0             16            30  data/train/masks/mask_0.png   
1        1              1            30  data/train/masks/mask_1.png   
2        2             38            30  data/train/masks/mask_2.png   
3        3             11            30  data/train/masks/mask_3.png   
4        4             38            30  data/train/masks/mask_4.png   

                      Patched_path Train/Test  
0  data/train/images/patched_0.png       Test  
1  data/train/images/patched_1.png       Test  
2  data/train/images/patched_2.png       Test  
3  data/train/images/patched_3.png       Test  
4  data/train/images/patched_4.png       Test  


Processing Images: 100%|██████████| 6000/6000 [00:40<00:00, 147.64it/s]

        Filename  Original_pred  Patched_pred  Defense_pred
0  patched_0.png             16            30            21
1  patched_1.png              1            30            30
2  patched_2.png             38            30            30
3  patched_3.png             11            30            30
4  patched_4.png             38            30            30
